<a href="https://colab.research.google.com/github/hUSsAin976-tech/ML-internship-at-FlyRank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

Lane: **AI Referral Opportunity** (freestyle). This notebook trains a model on top of the ML-07
baseline and asks one question: does combining several signals, fitted, beat the single
impression-volume rule the baseline uses — on the same data, the same split, the same metric?

Same data contract as ML-04/ML-07: `fact_content_daily_performance`, `month=2026-03` partition
only (never `_sample`), joined to `dim_content`. Label stays `has_ai_sessions_month` — an
**evidence variable**, not a forecast target (see ML-04's data-limits section) — used here to
fit and validate a resemblance score, exactly as it was used to validate the ML-04/ML-07 scores.

> Working with an AI assistant? Read `skills/README.md`, then load `training-honest-models` +
> `flyrank/flyrank-data` for this task.

## 1. Method choice and why

**The constraint carried in from ML-06/ML-07 doesn't go away just because this week's menu has
stronger tools on it.** `has_ai_sessions_month` positives are sparse warehouse-wide (30,177 of
78.8M daily rows) and thin within any one month/lane slice — that's why ML-04/ML-07 already ruled
out supervised classification metrics (AUC, accuracy) in favor of a validated ranking, checked
with **lift@K** against the observed base rate. That framing does not change this week. What
changes is *how the score is built*: instead of one hand-picked feature (impression volume,
ML-07's rule), I let a model combine all five contract features and see whether the combination
ranks better than volume alone, on the same lift@K metric.

**Method, from this week's toolkit, matched to the question shape:**

| Question shape (from the skill) | My question | Method |
|---|---|---|
| "yes/no with an observed label" | Does this content item show `has_ai_sessions_month = 1`? | Logistic Regression — readable coefficients, fits a linear combination of the same features the baseline already trusts |
| "which first?" ranking | Rank demand-worthy items by opportunity-resemblance | Evaluate the model's predicted probability at **precision@K / lift@K**, not accuracy — ranking needs scores, not labels |

**Logistic Regression is the primary model** — every coefficient is one number I can read and
defend, which matters more here than a fraction of a point of lift, and it directly extends
ML-07's transparent-rule philosophy (one score, explainable weights) rather than abandoning it.
**Random Forest is a second, deliberately bounded check** — non-linear interactions between
features (e.g. "high impressions AND top-10 position" might matter more together than either
alone) that a linear model can't see — plus it gives permutation importance as a second,
model-agnostic read on which features actually carry weight, independent of the sign and scale
assumptions built into the LR coefficients. **I stop at Random Forest and do not reach for
Gradient Boosting**, even though the menu allows it: five features and a thin positive count is
not enough signal for a boosted model's extra flexibility to earn its complexity — the
`training-honest-models` skill is explicit that added complexity has to win the comparison, not
just be available.

Numbers backing this choice — sparsity in this exact slice, printed below before any model is
fit.

In [1]:
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_month": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Same content_month view as ML-04/ML-07 (month=2026-03, GA4-available only), same five-feature
# contract, plus days_since_update_month (built, then deliberately unused in ML-07's rule after
# Signal B came back MIXED -- kept here as a candidate feature for the FITTED model to weigh on
# its own, since a model can use a weak marginal signal in combination even where a single-signal
# rule couldn't).
con.sql(f'''
    CREATE OR REPLACE TEMP VIEW content_month AS
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id)                                                  AS client_hash_id,
        SUM(f.gsc_impressions)                                                       AS total_gsc_impressions_month,
        AVG(CASE WHEN f.gsc_impressions > 0 THEN f.gsc_avg_position END)             AS avg_gsc_position_month,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)       AS days_with_impressions_month,
        MAX(CASE WHEN f.ga4_data_available IS TRUE AND f.sessions_ai > 0
                 THEN 1 ELSE 0 END)                                                  AS has_ai_sessions_month
    FROM {TABLES['fact_daily_month']} f
    WHERE f.ga4_data_available IS TRUE
    GROUP BY 1
''')

raw = con.sql(f'''
    SELECT
        cm.content_hash_id,
        cm.client_hash_id,
        cm.total_gsc_impressions_month,
        cm.avg_gsc_position_month,
        cm.days_with_impressions_month,
        cm.has_ai_sessions_month,
        dc.word_count,
        dc.content_type,
        DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-31') AS days_since_update_month
    FROM content_month cm
    JOIN {TABLES['dim_content']} dc USING (content_hash_id)
''').df()
raw["days_since_update_month"] = raw["days_since_update_month"].clip(lower=0)

# Demand-worthy slice -- identical filter to ML-04/ML-07 (impressions >= 100). This is the
# population both the baseline and the model are scored on below. Note this is NOT restricted to
# has_ai_sessions_month == 0: ML-07's delivered queue applies that extra filter because a page
# that already has AI sessions isn't a review candidate for the ACTION queue, but lift@K
# validation needs the positives in the evaluation set to measure anything at all -- this matches
# how ML-04 validated its quick score, not how ML-07 filtered its final output.
MIN_DEMAND_IMPRESSIONS = 100
df = raw[raw["total_gsc_impressions_month"] >= MIN_DEMAND_IMPRESSIONS].copy()

print(f"content_month, GA4-available: {len(raw):,} rows")
print(f"demand-worthy slice (impressions >= {MIN_DEMAND_IMPRESSIONS}): {len(df):,} rows")
print(f"positives (has_ai_sessions_month == 1): {int(df['has_ai_sessions_month'].sum()):,}")
print(f"base rate in this slice: {df['has_ai_sessions_month'].mean():.2%}")
print()
print(f"Why not accuracy/AUC as the headline metric: with a base rate this low, a model that")
print(f"predicts 'no' for everyone already scores {(1 - df['has_ai_sessions_month'].mean()):.1%} accuracy")
print(f"while being useless -- lift@K stays the metric, same as ML-04/ML-07.")
df.head()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

content_month, GA4-available: 90,489 rows
demand-worthy slice (impressions >= 100): 32,596 rows
positives (has_ai_sessions_month == 1): 2,808
base rate in this slice: 8.61%

Why not accuracy/AUC as the headline metric: with a base rate this low, a model that
predicts 'no' for everyone already scores 91.4% accuracy
while being useless -- lift@K stays the metric, same as ML-04/ML-07.


,content_hash_id,client_hash_id,total_gsc_impressions_month,avg_gsc_position_month,days_with_impressions_month,has_ai_sessions_month,word_count,content_type,days_since_update_month
6,content_6b0149a80607dac3,client_65de48885f4ef01b,1199.0,8.119012,17,1,1225,feedly article,34
7,content_62673eea26c31c17,client_65de48885f4ef01b,57145.0,6.814939,30,1,1278,feedly article,34
10,content_4c185d1c173cd53d,client_65de48885f4ef01b,278.0,10.399210,13,0,1223,feedly article,34
11,content_bd07be40ea0d5f54,client_65de48885f4ef01b,242.0,24.231436,15,1,1419,feedly article,34
15,content_40e28f4b41764012,client_65de48885f4ef01b,497.0,5.972594,10,0,1421,feedly article,34


## 2. Split design

**Grouped by `client_hash_id`, not random.** Content items from the same client share hidden
character — site template, existing SEO maturity, which AI tools happen to crawl that domain,
even the client's general topic authority — any of which could make items from the *same* client
look similar to a model for reasons that have nothing to do with the opportunity signal I'm
trying to measure. A random row-level split would let both Logistic Regression and Random Forest
partially memorize "which client is this" rather than learn the transferable pattern, and would
flatter both models relative to the baseline (which has no such memorization to exploit — it's a
hand-coded rule with nothing fit to the training data). The honest question is: *does the score
still work on a client the model never saw during training?*

`GroupShuffleSplit(test_size=0.25, random_state=42)`, one split, groups = `client_hash_id`.
75% of clients (and every row belonging to them) go to train; the remaining 25% of clients (and
every row belonging to them) go to test. I verify below that **zero clients appear in both
sides** — that check is the split's honesty receipt, not an assumption.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

X_groups = df["client_hash_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=X_groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

train_clients = set(train_df["client_hash_id"])
test_clients  = set(test_df["client_hash_id"])
overlap = train_clients & test_clients

print(f"train: {len(train_df):,} rows, {len(train_clients)} clients, "
      f"base rate {train_df['has_ai_sessions_month'].mean():.2%}")
print(f"test:  {len(test_df):,} rows, {len(test_clients)} clients, "
      f"base rate {test_df['has_ai_sessions_month'].mean():.2%}")
print(f"\nclients appearing in BOTH train and test: {len(overlap)}  (0 means the grouped split held)")
assert len(overlap) == 0, "Group leakage: a client_hash_id appears on both sides of the split."


train: 27,724 rows, 22 clients, base rate 9.48%
test:  4,872 rows, 8 clients, base rate 3.67%

clients appearing in BOTH train and test: 0  (0 means the grouped split held)


## 3. Train + compare vs my baseline

**Same test set, same metric, for all three scores below** — the baseline rule, Logistic
Regression, and Random Forest are all scored on the `test_df` slice built in Section 2 (clients
the model never trained on), at the same `K`, against the same printed base rate.

- **Baseline** — ML-07's rule exactly: percentile rank of `total_gsc_impressions_month`, higher
  volume = higher score. Recomputed fresh *within the test set only* (it was never "trained" —
  a rule has nothing to fit — but scoring it on the same held-out rows as the models keeps the
  comparison fair rather than reusing ML-07's full-population ranking).
- **Logistic Regression** — fit on `train_df`'s five features (impressions, position,
  days-with-impressions, word count, content type) plus `days_since_update_month`, scored by
  predicted probability on `test_df`.
- **Random Forest** — same features, same train/test split, scored by predicted probability on
  `test_df`.

`K = max(50, 5% of test_df)`, matching the `K` convention from ML-04's validation.

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUMERIC_FEATURES = [
    "total_gsc_impressions_month", "avg_gsc_position_month",
    "days_with_impressions_month", "word_count", "days_since_update_month",
]
CATEGORICAL_FEATURES = ["content_type"]
TARGET = "has_ai_sessions_month"

def prep(frame):
    # Impute with an explicit has_-flag first, per the flyrank-data skill's missingness gotcha --
    # missingness follows content_type, so a blind fillna(0) would silently encode a content-type
    # signal into the imputed value. The flag keeps that information honest instead.
    out = frame.copy()
    for col in ["word_count", "avg_gsc_position_month", "days_since_update_month"]:
        out[f"has_{col}"] = out[col].notna().astype(int)
        # Convert to float before fillna to handle potential float medians and NaN values
        out[col] = out[col].astype(float).fillna(out[col].median())
    return out

train_p = prep(train_df)
test_p  = prep(test_df)
FLAG_FEATURES = [f"has_{c}" for c in ["word_count", "avg_gsc_position_month", "days_since_update_month"]]
ALL_NUMERIC = NUMERIC_FEATURES + FLAG_FEATURES

X_train = train_p[ALL_NUMERIC + CATEGORICAL_FEATURES]
y_train = train_p[TARGET]
X_test  = test_p[ALL_NUMERIC + CATEGORICAL_FEATURES]
y_test  = test_p[TARGET]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), ALL_NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

log_reg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
]).fit(X_train, y_train)

rand_forest = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )),
]).fit(X_train, y_train)

def lift_at_k(scores, y, k):
    order = np.argsort(-np.asarray(scores))
    top_y = np.asarray(y)[order][:k]
    precision = top_y.mean()
    base = np.asarray(y).mean()
    return precision, precision / base

base_rate_test = y_test.mean()
K = max(50, int(round(0.05 * len(test_p))))

baseline_score = test_p["total_gsc_impressions_month"].rank(pct=True).values
logreg_score   = log_reg.predict_proba(X_test)[:, 1]
rf_score       = rand_forest.predict_proba(X_test)[:, 1]

rows = []
for name, score in [
    ("Baseline (impression rank only, ML-07 rule)", baseline_score),
    ("Logistic Regression", logreg_score),
    ("Random Forest", rf_score),
]:
    precision, lift = lift_at_k(score, y_test.values, K)
    rows.append({"model": name, "K": K, "base_rate": base_rate_test,
                 "precision@K": precision, "lift@K": lift})

comparison = pd.DataFrame(rows).set_index("model")
comparison["base_rate"] = comparison["base_rate"].map(lambda v: f"{v:.2%}")
comparison["precision@K"] = comparison["precision@K"].map(lambda v: f"{v:.2%}")
comparison["lift@K"] = comparison["lift@K"].map(lambda v: f"{v:.2f}x")
comparison

,K,base_rate,precision@K,lift@K
model,,,,
"Baseline (impression rank only, ML-07 rule)",244,3.67%,11.07%,3.01x
Logistic Regression,244,3.67%,18.44%,5.02x
Random Forest,244,3.67%,13.93%,3.79x


## 4. Errors and interpretation

Three things to check before believing the table above: what each model leans on, whether the
top feature is plausible or "too good," and a handful of concrete cases where the ranking gets it
wrong.

In [5]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

# --- 1. Logistic Regression coefficients (standardized -> directly comparable) ---
feature_names = (
    ALL_NUMERIC
    + list(log_reg.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
)
coefs = pd.Series(log_reg.named_steps["clf"].coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
print("--- Logistic Regression coefficients (standardized numeric features) ---")
print(coefs.round(3).to_string())

# --- 2. Random Forest permutation importance on the held-out test set ---
# Uses average_precision as the internal scorer for ranking features against each other -- a
# diagnostic choice for THIS cell only. The headline metric the models are judged by stays
# lift@K/precision@K in Section 3; this is not a substitute for that comparison.
perm = permutation_importance(
    rand_forest, X_test, y_test, scoring="average_precision",
    n_repeats=20, random_state=42, n_jobs=-1,
)
perm_table = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("\n--- Random Forest permutation importance (mean drop in average_precision) ---")
print(perm_table.round(4).to_string())

print(f"\nSanity check: top feature is '{perm_table.index[0]}'. Impressions leading is expected --")
print(f"it's the same independent, non-leaky Search Console signal the ML-07 baseline already")
print(f"trusts. If a feature adjacent to the label (or a near-duplicate of it) had topped this")
print(f"list instead, that would be the 'suspiciously perfect' signal to stop and investigate.")

# --- 3. Where does days_since_update_month land? (the Signal B question, revisited) ---
print(f"\ndays_since_update_month: LR coefficient rank "
      f"{list(coefs.index).index('days_since_update_month') + 1} of {len(coefs)}, "
      f"permutation importance rank {list(perm_table.index).index('days_since_update_month') + 1} "
      f"of {len(perm_table)} -- consistent with ML-07's MIXED verdict: the model, given the chance "
      f"to use it anyway, still doesn't lean on it much.")

# --- 4. Concrete disagreement cases: model right where baseline is wrong, and vice versa ---
test_view = test_p[["content_hash_id", "total_gsc_impressions_month", "avg_gsc_position_month",
                     "content_type", TARGET]].copy()
test_view["baseline_score"] = baseline_score
test_view["logreg_score"] = logreg_score
baseline_topk = set(test_view.sort_values("baseline_score", ascending=False).head(K)["content_hash_id"])
logreg_topk   = set(test_view.sort_values("logreg_score", ascending=False).head(K)["content_hash_id"])

model_only = test_view[test_view["content_hash_id"].isin(logreg_topk - baseline_topk)]
baseline_only = test_view[test_view["content_hash_id"].isin(baseline_topk - logreg_topk)]

print(f"\nin LR's top-{K} but NOT baseline's: {len(model_only)} items, "
      f"{model_only[TARGET].mean():.1%} are true positives")
print(f"in baseline's top-{K} but NOT LR's: {len(baseline_only)} items, "
      f"{baseline_only[TARGET].mean():.1%} are true positives")

print("\n--- 3 concrete cases ---")
print("Case 1 -- LR catches a true positive the baseline misses (if any exist):")
print(model_only[model_only[TARGET] == 1].head(1)[
    ["content_hash_id", "total_gsc_impressions_month", "avg_gsc_position_month", "content_type"]
].to_string(index=False) or "none in this K -- LR did not out-catch the baseline here")

print("\nCase 2 -- baseline catches a true positive LR misses (if any exist):")
print(baseline_only[baseline_only[TARGET] == 1].head(1)[
    ["content_hash_id", "total_gsc_impressions_month", "avg_gsc_position_month", "content_type"]
].to_string(index=False) or "none in this K -- baseline did not out-catch LR here")

print("\nCase 3 -- a false positive in LR's top-K worth a second look:")
lr_fp = model_only[model_only[TARGET] == 0].head(1)
print(lr_fp[["content_hash_id", "total_gsc_impressions_month", "avg_gsc_position_month",
             "content_type"]].to_string(index=False) if len(lr_fp) else "none -- LR's extra picks were all true positives")


--- Logistic Regression coefficients (standardized numeric features) ---
content_type_feedly article        1.452
content_type_keyword article      -1.008
days_with_impressions_month        0.877
avg_gsc_position_month             0.460
total_gsc_impressions_month        0.062
has_word_count                     0.053
word_count                         0.041
days_since_update_month            0.030
content_type_comparison article    0.005
has_avg_gsc_position_month         0.000
has_days_since_update_month        0.000

--- Random Forest permutation importance (mean drop in average_precision) ---
days_with_impressions_month    0.0258
content_type                   0.0209
avg_gsc_position_month         0.0120
total_gsc_impressions_month    0.0043
days_since_update_month        0.0026
has_days_since_update_month    0.0000
has_avg_gsc_position_month     0.0000
has_word_count                -0.0001
word_count                    -0.0142

Sanity check: top feature is 'days_with_impressions_mo

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.